``
Spark UI – Opis zakładek:

- Jobs -	Lista zadań wykonanych przez Spark. Każde Job to logiczne zapytanie użytkownika, dzieli się na Stages.
- Stages -	Szczegóły wykonania każdej fazy zadania (np. map, reduce). Zawiera czas wykonania, ilość przetwarzanych rekordów, shuffle.
- Storage	- Informacje o danych przechowywanych w pamięci (np. cache/persist). Pokazuje ile danych i gdzie zostały zbuforowane.
- Executors -	Pokazuje informacje o każdym executorze (np. użycie pamięci, liczba przetworzonych tasków).
- SQL / DataFrame	 - Historia zapytań SQL/DataFrame API – czas wykonania, plan fizyczny/logiczny.
- Dystrybucja danych -	Widać ją głównie w zakładkach Stages (np. liczba partycji, shuffle) oraz SQL/Dataframe (plan wykonania z podziałem na etapy).

## ZADANIE 2 – Bucketing vs PartitionBy


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand

spark = SparkSession.builder \
    .appName("Bucketing Example") \
    .enableHiveSupport() \
    .getOrCreate()

df = spark.range(0, 100000).withColumn("user_id", (rand() * 1000).cast("int"))

#Bucketing
spark.sql("DROP TABLE IF EXISTS users_bucketed")
df.write \
    .bucketBy(10, "user_id") \
    .sortBy("user_id") \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("users_bucketed")


#Partitioning
spark.sql("DROP TABLE IF EXISTS users_partitioned")
df.write \
    .partitionBy("user_id") \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("users_partitioned")


## ZADANIE 3 – ANALYZE TABLE (Statystyki)


In [0]:
%sql
CREATE OR REPLACE TABLE sample_data AS
SELECT id, rand() as value, floor(rand() * 100) as category
FROM range(10000);

-- ANALYZE TABLE – statystyki ogólne
ANALYZE TABLE sample_data COMPUTE STATISTICS;

-- ANALYZE TABLE – statystyki dla wszystkich kolumn
ANALYZE TABLE sample_data COMPUTE STATISTICS FOR COLUMNS id, value, category;

-- Sprawdzenie statystyk
DESCRIBE TABLE EXTENDED sample_data;

DESCRIBE DETAIL sample_data;


format,id,name,description,location,createdAt,lastModified,partitionColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,a8573693-b6f2-4eca-baf9-e4c2e48b8d03,spark_catalog.default.sample_data,null,dbfs:/user/hive/warehouse/sample_data,2025-05-28T07:44:19.927+0000,2025-05-28T07:44:23.000+0000,List(),8,142386,Map(),1,2,"List(appendOnly, invariants)",Map()


- COMPUTE STATISTICS – tworzy statystyki ogólne (liczba rekordów, liczba plików).

- FOR COLUMNS – tworzy statystyki dla konkretnych kolumn (min, max, distinct, nulls).

Statystyki są używane przez optymalizator zapytań Spark do lepszego planowania.